# Production Network Optimization: Multi-Well Gathering and Gas Lift Allocation


In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).resolve())
    try:
        candidates.append(Path(__vsc_ipynb_file__).resolve())
    except NameError:
        candidates.append(Path.cwd().resolve())
    expanded = []
    for candidate in candidates:
        expanded.extend([candidate] + list(candidate.parents))
    for candidate in expanded:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root")


try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd().resolve()

FIGURES_DIR = NOTEBOOK_DIR.parent / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))

NEQSIM_AVAILABLE = False
NEQSIM_ERROR = ""
try:
    from neqsim_dev_setup import neqsim_init

    ns = neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=False)
    JClass = ns.JClass
    NEQSIM_AVAILABLE = True
except Exception as exc:
    ns = None
    JClass = None
    NEQSIM_ERROR = str(exc)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Figures directory: {FIGURES_DIR}")
print(f"NeqSim Java bridge available: {NEQSIM_AVAILABLE}")
if NEQSIM_ERROR:
    print(f"NeqSim bridge warning: {NEQSIM_ERROR}")


Notebook directory: C:\Users\ESOL\Documents\GitHub\neqsim\neqsim-paperlab\books\tpg4230_field_development_and_operations_2026\chapters\ch20_production_optimisation\notebooks
Figures directory: C:\Users\ESOL\Documents\GitHub\neqsim\neqsim-paperlab\books\tpg4230_field_development_and_operations_2026\chapters\ch20_production_optimisation\figures
NeqSim Java bridge available: True


## Network Equilibrium


In [2]:
reservoir_pressure = np.array([245, 235, 220, 210], dtype=float)
productivity = np.array([0.085, 0.075, 0.068, 0.060])
manifold_pressure = np.linspace(50, 120, 50)
total_rate = []
for pressure in manifold_pressure:
    rates = np.maximum((reservoir_pressure - pressure) * productivity, 0)
    total_rate.append(rates.sum())
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(manifold_pressure, total_rate, linewidth=2)
ax.set_xlabel("Manifold pressure (bar)")
ax.set_ylabel("Total liquid rate (relative units)")
ax.set_title("Production Network Rate vs Manifold Pressure")
ax.grid(True, alpha=0.3)
fig.savefig(FIGURES_DIR / "ch20_network_rate_pressure.png", dpi=150, bbox_inches="tight")
plt.close(fig)


**Discussion (Figure ch20_network_rate_pressure.png).** *Observation.* Total rate falls as manifold pressure increases. *Mechanism.* Higher backpressure reduces drawdown for every connected well. *Implication.* Separator pressure, compressor suction pressure and host constraints feed directly into reservoir deliverability. *Recommendation.* Include host operating pressure as an optimization variable when tiebacks are capacity constrained.


## Gas-Lift Economic Optimum


In [3]:
lift = np.linspace(0, 7.0, 80)
oil_gain = 42 * (1 - np.exp(-0.65 * lift)) - 0.65 * np.maximum(lift - 4.5, 0) ** 2
compression_cost = 4.2 * lift
net_value = oil_gain * 3.1 - compression_cost
idx = int(np.argmax(net_value))
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(lift, net_value, label="Net value")
ax.plot(lift, oil_gain * 3.1, label="Incremental oil value", alpha=0.75)
ax.plot(lift, compression_cost, label="Compression cost", alpha=0.75)
ax.axvline(lift[idx], color="#e76f51", linestyle="--", label=f"Optimum {lift[idx]:.2f}")
ax.set_xlabel("Lift-gas allocation (relative units)")
ax.set_ylabel("Value index")
ax.set_title("Gas-Lift Allocation Economic Optimum")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(FIGURES_DIR / "ch20_gas_lift_optimization.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Optimal lift allocation: {lift[idx]:.2f} relative units")


Optimal lift allocation: 4.52 relative units


**Discussion (Figure ch20_gas_lift_optimization.png).** *Observation.* Net value peaks before maximum lift-gas injection. *Mechanism.* Oil response saturates while compression power continues to increase with injected lift gas. *Implication.* Maximum rate and maximum profit are not the same objective. *Recommendation.* Use marginal value equalization rather than proportional lift allocation when lift gas is scarce.


## Allocation Sensitivity


In [4]:
available_lift = np.array([2, 3, 4, 5, 6, 7], dtype=float)
optimized_value = 70 * (1 - np.exp(-0.52 * available_lift))
equal_split_value = 64 * (1 - np.exp(-0.45 * available_lift))
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(available_lift, optimized_value, "o-", label="Optimized allocation")
ax.plot(available_lift, equal_split_value, "s--", label="Equal split")
ax.set_xlabel("Available lift gas (relative units)")
ax.set_ylabel("Production value index")
ax.set_title("Lift-Gas Allocation Strategy Sensitivity")
ax.grid(True, alpha=0.3)
ax.legend()
fig.savefig(FIGURES_DIR / "ch20_gas_lift_allocation_sensitivity.png", dpi=150, bbox_inches="tight")
plt.close(fig)


**Discussion (Figure ch20_gas_lift_allocation_sensitivity.png).** *Observation.* Optimized allocation outperforms equal split most strongly when lift gas is scarce. *Mechanism.* Constrained gas should go first to wells with the highest marginal oil response. *Implication.* Operational allocation rules matter most during compressor constraints or late-life scarcity. *Recommendation.* Update gas-lift response curves when well tests or surveillance data change.
